In [1]:
import pandas as pd
import glob
import os
from sklearn.model_selection import train_test_split



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Take first 20 files from Metadata

In [3]:
metadata_path = "/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/*.csv"

# Collect all metadata files
metadata_files = glob.glob(metadata_path)

# Take only the first 20 files
selected_files = metadata_files[:20]

print("Selected files:", selected_files)


Selected files: ['/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00003.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00014.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00007.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00019.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00012.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00009.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00013.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00011.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00002.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00015.csv', '/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata/p00004.csv', '/content/drive

Combine the files together to gather all the python related data

In [4]:
df_list = []
for file in selected_files:
    df = pd.read_csv(file)
    # Optional: filter only Python solutions
    df = df[df["language"] == "Python"]
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

# Save to one file
combined_df.to_csv("python_metadata_first10.csv", index=False)

print("Combined dataset shape:", combined_df.shape)


Combined dataset shape: (13696, 12)


In [5]:
combined_df

,submission_id,problem_id,user_id,date,language,original_language,filename_ext,status,cpu_time,memory,code_size,accuracy
0,s257278998,p00003,u539753516,1531568674,Python,Python3,py,Accepted,40,5592,166,1/1
1,s655328803,p00003,u525366883,1535372099,Python,Python,py,Runtime Error,0,0,135,0/1
2,s663133214,p00003,u525366883,1535372136,Python,Python,py,Wrong Answer,10,4636,156,0/1
3,s161085846,p00003,u525366883,1535372250,Python,Python,py,Runtime Error,0,0,177,0/1
4,s004493480,p00003,u525366883,1535372447,Python,Python,py,Accepted,10,4644,199,1/1
...,...,...,...,...,...,...,...,...,...,...,...,...
13691,s945518269,p00006,u853158149,1521964036,Python,Python3,py,Accepted,20,5540,21,1/1
13692,s557060130,p00006,u281836941,1514712140,Python,Python3,py,Accepted,20,5544,21,1/1
13693,s005962645,p00006,u079141094,1467250536,Python,Python3,py,Accepted,30,7396,30,1/1
13694,s996034886,p00006,u822200871,1447520475,Python,Python3,py,Accepted,30,7368,57,1/1


Label Buggy vs. Clean
------------------------------
From the combined metadata:
- Mark submissions as buggy = 1 if status != "Accepted".
- Mark submissions as buggy = 0 if status == "Accepted"


In [6]:
combined_df["buggy"] = combined_df["status"].apply(lambda x: 1 if x != "Accepted" else 0)
combined_df

,submission_id,problem_id,user_id,date,language,original_language,filename_ext,status,cpu_time,memory,code_size,accuracy,buggy
0,s257278998,p00003,u539753516,1531568674,Python,Python3,py,Accepted,40,5592,166,1/1,0
1,s655328803,p00003,u525366883,1535372099,Python,Python,py,Runtime Error,0,0,135,0/1,1
2,s663133214,p00003,u525366883,1535372136,Python,Python,py,Wrong Answer,10,4636,156,0/1,1
3,s161085846,p00003,u525366883,1535372250,Python,Python,py,Runtime Error,0,0,177,0/1,1
4,s004493480,p00003,u525366883,1535372447,Python,Python,py,Accepted,10,4644,199,1/1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13691,s945518269,p00006,u853158149,1521964036,Python,Python3,py,Accepted,20,5540,21,1/1,0
13692,s557060130,p00006,u281836941,1514712140,Python,Python3,py,Accepted,20,5544,21,1/1,0
13693,s005962645,p00006,u079141094,1467250536,Python,Python3,py,Accepted,30,7396,30,1/1,0
13694,s996034886,p00006,u822200871,1447520475,Python,Python3,py,Accepted,30,7368,57,1/1,0


Split into Train/Test

In [7]:
train_df, test_df = train_test_split(combined_df, test_size=0.2, random_state=42)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)


Train size: (10956, 13)
Test size: (2740, 13)


Prepare for Bug Prediction

In [8]:
X_train = train_df[["cpu_time", "memory"]].fillna(0)
y_train = train_df["buggy"]

X_test = test_df[["cpu_time", "memory"]].fillna(0)
y_test = test_df["buggy"]


Prepare for Auto-Fix

In [9]:
buggy_fixed_pairs = []
grouped = combined_df.groupby("problem_id")

for pid, group in grouped:
    buggy = group[group["buggy"] == 1]
    fixed = group[group["buggy"] == 0]
    for _, b in buggy.iterrows():
        for _, f in fixed.iterrows():
            buggy_fixed_pairs.append({"buggy": b["submission_id"], "fixed": f["submission_id"], "problem_id": f["problem_id"]})

pairs_df = pd.DataFrame(buggy_fixed_pairs)
pairs_df.to_csv("python_bugfix_pairs.csv", index=False)